In [144]:
# Reproducible EDA setup (same behavior in Cursor and VS Code)
import os
import sys
import platform
import hashlib
from pathlib import Path

import pandas as pd
import numpy as np

# Keep notebook outputs deterministic where randomness is used
SEED = 42
np.random.seed(SEED)

# Resolve project root from notebook location, not current working directory
# notebook/eda.ipynb -> project root is parent of notebook/
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "notebook").exists() and (PROJECT_ROOT / "src").exists():
    pass
else:
    PROJECT_ROOT = Path.cwd().parent if (Path.cwd().name == "notebook") else Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd()

NOTEBOOK_DIR = PROJECT_ROOT / "notebook"
DATA_DIR = PROJECT_ROOT / "data"

# Set exactly one dataset path here (recommended).
# Example: DATASET_PATH = DATA_DIR / "raw" / "train.csv"
DATASET_PATH = None

# Optional: if you know expected hash from VS Code, paste it here.
EXPECTED_MD5 = None


def file_md5(path: Path) -> str:
    hasher = hashlib.md5()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


def auto_pick_dataset() -> Path | None:
    candidates = []
    for folder in [DATA_DIR, DATA_DIR / "raw", DATA_DIR / "processed", PROJECT_ROOT]:
        if folder.exists():
            candidates.extend(sorted(folder.glob("*.csv")))
            candidates.extend(sorted(folder.glob("*.parquet")))
    return candidates[0] if candidates else None


if DATASET_PATH is None:
    DATASET_PATH = auto_pick_dataset()

print("=== Runtime ===")
print("python_executable:", sys.executable)
print("python_version   :", sys.version.replace("\n", " "))
print("platform         :", platform.platform())
print("cwd              :", os.getcwd())

print("\n=== Package versions ===")
print("pandas:", pd.__version__)
print("numpy :", np.__version__)

print("\n=== Project paths ===")
print("project_root:", PROJECT_ROOT.resolve())
print("notebook_dir:", NOTEBOOK_DIR.resolve())
print("data_dir    :", DATA_DIR.resolve())

if DATASET_PATH is None:
    raise FileNotFoundError(
        "No dataset found. Put a .csv/.parquet file in data/ (or data/raw) or set DATASET_PATH manually."
    )

DATASET_PATH = Path(DATASET_PATH)
if not DATASET_PATH.is_absolute():
    DATASET_PATH = (PROJECT_ROOT / DATASET_PATH).resolve()

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset path does not exist: {DATASET_PATH}")

CURRENT_MD5 = file_md5(DATASET_PATH)
print("\n=== Dataset identity ===")
print("dataset_path:", DATASET_PATH)
print("size_bytes  :", DATASET_PATH.stat().st_size)
print("md5         :", CURRENT_MD5)

if EXPECTED_MD5 is not None and CURRENT_MD5 != EXPECTED_MD5:
    raise ValueError(
        f"Dataset mismatch! expected={EXPECTED_MD5}, current={CURRENT_MD5}. Use same file in both editors."
    )

print("\nSeed fixed to:", SEED)

=== Runtime ===
python_executable: c:\Users\USER\anaconda3\python.exe
python_version   : 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]
platform         : Windows-11-10.0.22631-SP0
cwd              : c:\RAG 1\notebook

=== Package versions ===
pandas: 2.2.2
numpy : 1.26.4

=== Project paths ===
project_root: C:\RAG 1
notebook_dir: C:\RAG 1\notebook
data_dir    : C:\RAG 1\data

=== Dataset identity ===
dataset_path: c:\RAG 1\data\processed\full_tweets_for_prediction.csv
size_bytes  : 17028
md5         : bae366db56cc1310e0fc5f6851244715

Seed fixed to: 42


In [145]:
# Load data consistently and run deterministic starter EDA
if DATASET_PATH.suffix.lower() == ".csv":
    df = pd.read_csv(DATASET_PATH)
elif DATASET_PATH.suffix.lower() == ".parquet":
    df = pd.read_parquet(DATASET_PATH)
else:
    raise ValueError(f"Unsupported file type: {DATASET_PATH.suffix}. Use .csv or .parquet")

print("shape:", df.shape)
print("columns:", len(df.columns))


shape: (93, 7)
columns: 7


# Ensure df exists even if cells are run out of order
if "df" not in globals():
    if "DATASET_PATH" not in globals() or DATASET_PATH is None:
        raise NameError("DATASET_PATH is not defined. Run the setup cell first.")

    if DATASET_PATH.suffix.lower() == ".csv":
        df = pd.read_csv(DATASET_PATH)
    elif DATASET_PATH.suffix.lower() == ".parquet":
        df = pd.read_parquet(DATASET_PATH)
    else:
        raise ValueError(f"Unsupported file type: {DATASET_PATH.suffix}. Use .csv or .parquet")

# understand the structure

In [ ]:
df.shape
df.columns
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,119237,105834,True,2017-10-11 06:55:44+00:00,@AppleSupport causing the reply to be disregar...,119236,NaN
1,119238,ChaseSupport,False,2017-10-11 13:25:49+00:00,@105835 Your business means a lot to us. Pleas...,NaN,119239.0
2,119239,105835,True,2017-10-11 13:00:09+00:00,@76328 I really hope you all change but I'm su...,119238,NaN
3,119240,VirginTrains,False,2017-10-10 15:16:08+00:00,@105836 LiveChat is online at the moment - htt...,119241,119242.0
4,119241,105836,True,2017-10-10 15:17:21+00:00,@VirginTrains see attached error message. I've...,119243,119240.0


# Data types and missing values overview



In [ ]:

dtype_df = df.dtypes.rename("dtype").to_frame()
missing_df = (
    df.isna().sum().rename("missing_count").to_frame()
    .assign(missing_pct=lambda x: (x["missing_count"] / len(df) * 100).round(2))
)
overview = dtype_df.join(missing_df).sort_values(["missing_count", "dtype"], ascending=[False, True])
display(overview)


,dtype,missing_count,missing_pct
response_tweet_id,object,28,30.11
in_response_to_tweet_id,float64,25,26.88
inbound,bool,0,0.00
tweet_id,int64,0,0.00
author_id,object,0,0.00
created_at,object,0,0.00
text,object,0,0.00


# Numeric summary is deterministic

In [148]:

display(df.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
tweet_id,93.0,NaN,NaN,NaN,119285.451613,28.314045,119237.0,119262.0,119285.0,119309.0,119335.0
author_id,93,42,AppleSupport,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
inbound,93,2,True,49,NaN,NaN,NaN,NaN,NaN,NaN,NaN
created_at,93,93,2017-10-11 06:55:44+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
text,93,93,@AppleSupport causing the reply to be disregar...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
response_tweet_id,65,65,119236,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
in_response_to_tweet_id,68.0,NaN,NaN,NaN,119285.676471,28.910795,119239.0,119259.75,119284.5,119311.5,119334.0


# who is speaking ?

In [149]:
df["inbound"].value_counts()

inbound
True     49
False    44
Name: count, dtype: int64

# author analysis

In [150]:
df["author_id"].value_counts().head(10)

author_id
AppleSupport       13
Tesco               8
SpotifyCares        8
VirginTrains        4
105847              4
105840              4
105855              4
105848              3
105836              3
British_Airways     3
Name: count, dtype: int64

# text analysis , help for RAG

In [151]:
df["text_length"] = df["text"].astype(str).apply(len)

df["text_length"].describe()

count     93.000000
mean     118.559140
std       33.668932
min       32.000000
25%       93.000000
50%      132.000000
75%      146.000000
max      170.000000
Name: text_length, dtype: float64

# CONVERSATION STRUCTURE 

In [152]:
df[["tweet_id", "response_tweet_id", "text"]].head(10)

,tweet_id,response_tweet_id,text
0,119237,119236,@AppleSupport causing the reply to be disregar...
1,119238,NaN,@105835 Your business means a lot to us. Pleas...
2,119239,119238,@76328 I really hope you all change but I'm su...
3,119240,119241,@105836 LiveChat is online at the moment - htt...
4,119241,119243,@VirginTrains see attached error message. I've...
5,119243,119244,"@105836 Have you tried from another device, Mi..."
6,119244,119245,"@VirginTrains yep, I've tried laptop too sever..."
7,119245,NaN,"@105836 It's working OK from here, Miriam. Doe..."
8,119242,119240,@VirginTrains I still haven't heard &amp; the ...
9,119246,119242,@105836 That's what we're here for Miriam 😊 T...


# Usable conservation , only tweets with replies are useful for RAG 

In [153]:
df[df["response_tweet_id"].notnull()].head()


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,text_length
0,119237,105834,True,2017-10-11 06:55:44+00:00,@AppleSupport causing the reply to be disregar...,119236,NaN,109
2,119239,105835,True,2017-10-11 13:00:09+00:00,@76328 I really hope you all change but I'm su...,119238,NaN,86
3,119240,VirginTrains,False,2017-10-10 15:16:08+00:00,@105836 LiveChat is online at the moment - htt...,119241,119242.0,147
4,119241,105836,True,2017-10-10 15:17:21+00:00,@VirginTrains see attached error message. I've...,119243,119240.0,127
5,119243,VirginTrains,False,2017-10-10 15:25:14+00:00,"@105836 Have you tried from another device, Mi...",119244,119241.0,54
